In [ ]:
using LinearAlgebra

In [ ]:
function hat(v)
    return [0 -v[3] v[2];
            v[3] 0 -v[1];
            -v[2] v[1] 0]
end

H = [zeros(1,3); I];
T = [1  zeros(1,3);
     zeros(3,1) -I];

function L(q)
    return [q[1]          -q[2:4]';
            q[2:4]    q[1]*I + hat(q[2:4])]
end

function R(q)
    return [q[1]          -q[2:4]';
            q[2:4]    q[1]*I - hat(q[2:4])]
end

function G(q)
    return L(q)*H
end

function Q(q)
    return H'*L(q)*R(q)'*H
end

In [ ]:
#model parameters
J = Diagonal([0.99; 1.01; 2])
Jd = 0.1*I
Cd = 0.1

In [ ]:
#dynamics
function dynamics(x)
    q = x[1:4]
    ω = x[5:7]
    ωd = x[8:10]

    τd = Cd*(ωd-ω)
    
    q̇ = 0.5*G(q)*ω
    ω̇ = J\(τd - hat(ω)*J*ω)
    ω̇d = -Jd\(τd + hat(ω)*Jd*ωd)

    ẋ = [q̇; ω̇; ω̇d]
end

In [ ]:
#Classic RK4 integrator: https://en.wikipedia.org/wiki/Runge%E2%80%93Kutta_methods
function rkstep(x)
    f1 = dynamics(x)
    f2 = dynamics(x + 0.5*h*f1)
    f3 = dynamics(x + 0.5*h*f2)
    f4 = dynamics(x + h*f3)
    xn = x + (h/6.0)*(f1 + 2*f2 + 2*f3 + f4)
    xn[1:4] .= xn[1:4]/norm(xn[1:4]) #re-normalize quaternion
    return xn
end

In [ ]:
h = 0.1 #time step
n = 2000 #number of time steps
tf = n*h #final time

#sample random angular velocity
q0 = [1; 0; 0; 0]
ω0 = [3; 0; 0] + 0.1*randn(3);
ωd0 = ω0;
x0 = [q0; ω0; ωd0];

In [ ]:
#Simulate n time steps
xhist = zeros(10,n)
xhist[:,1] .= x0
for k = 1:(n-1)
    xhist[:,k+1] = rkstep(xhist[:,k])
end

In [ ]:
using MeshCat, GeometryBasics, CoordinateTransformations, Rotations

In [ ]:
vis = Visualizer()

In [ ]:
render(vis)

In [ ]:
setobject!(vis[:box1], Rect(Vec(-0.3, -0.1, -0.1), Vec(0.6, 0.2, 0.2)))

In [ ]:
anim = Animation(vis)

for t = 1:n
    atframe(anim, t) do
        settransform!(vis[:box1], LinearMap(QuatRotation(xhist[1,t], xhist[2,t], xhist[3,t], xhist[4,t])))
    end
end

setanimation!(vis, anim)

In [ ]:
using PythonPlot

In [ ]:
plot(xhist[5,:])
plot(xhist[6,:])
plot(xhist[7,:])

In [ ]:
T = zeros(n)
for k = 1:n
    ω = xhist[5:7,k];
    ωd = xhist[8:10,k];
    T[k] = 0.5*ω'*J*ω + 0.5*ωd'*Jd*ωd
end

In [ ]:
plot(T)

In [ ]:
hn = zeros(3,n)
for k = 1:n
    hn[:,k] .= Q(xhist[1:4,k])*(J*xhist[5:7,k] + Jd*xhist[8:10,k])
end

In [ ]:
plot(hn[1,:])
plot(hn[2,:])
plot(hn[3,:])